# 00 · Synthetic data — checking notebook (dev only, temporary)

**What this is for.** Review the synthetic feature mart in one pass: is it a plausible retail book,
do the mechanics behave as specified, do the traps and the oracle work? The spec is
`docs/components/test_harness.md`; the review guide, with what "right" looks like for each section,
is `synthetic/REVIEW.md`. The guided EDA notebook replaces this one once EDA exists.

**How to use it.**
1. Generate the canonical dataset and its check tables once (about 6 minutes, plus 10–15 for the
   checks): `python -m synthetic --preset canonical --out data/01_raw/synthetic_canonical --checks`.
2. Restart the kernel and run all cells (a few minutes).
3. Section 1 first: every check row must pass. A failing row is a generator bug, never a tolerance
   to widen. Then sections 2–10, with the review guide alongside.

Every table says what its values are (share, mean, median…) in the line printed above it or in its
column names. Amounts are in AED a month. The oracle columns are hidden truth for checking only;
they are never features. Charts show aggregates; most use a fixed 20% client sample (client ids
divisible by 5) to keep memory low.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq

import synthetic
from synthetic.checks.suite import read_checks
from synthetic.mechanisms.traps import trap_summary
from synthetic.output import load_truth, read_output
from synthetic.truth.oracle import auc, realised_target, window_probability

# The dataset lives next to the repo, never relative to the working directory.
DATA = Path(synthetic.__file__).resolve().parents[3] / "data" / "01_raw" / "synthetic_canonical"
truth = load_truth(str(DATA))  # fails loud if the dataset is older than the installed generator
spec = truth["spec"]
ORDER = list(spec["subsegments"])  # wealthiest first
PROFILES = ["private", "priority", "affluent", "mass"]
PROFILE_OF = {s: v["profile"] for s, v in spec["subsegments"].items()}
MAY = pd.Timestamp("2025-05-01").date()  # a month without seasonal effects, bonuses or school fees
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_rows", 200)
print(f"generator {truth['generator_version']} · seed {truth['seed']}")
print(f"{truth['n_clients']:,} clients · {truth['n_months']} months from {truth['first_month']}")
print({k: v for k, v in truth["versions"].items()})

generator 0.4.0 · seed 0
100,000 clients · 24 months from 2024-10
{'numpy': '2.3.5', 'scipy': '1.16.2', 'pandas': '2.2.3', 'pyarrow': '19.0.1'}


## 1. Check tables
Every family's rows, from the spec, written independently of the generator, as saved by
`--checks` (each family runs in its own process: at 100k one family needs several GB). `failed`
must be 0 everywhere. The mart family reads every column, so it runs on the 20% client sample.

In [2]:
tables = read_checks(str(DATA))
summary = pd.DataFrame(
    [
        (name, len(t), int(t["passed"].sum()), int((~t["passed"]).sum()), int(t["rule"].sum()))
        for name, t in tables.items()
    ],
    columns=["family", "rows", "passed", "failed", "rules (tolerance 0)"],
)
print("Check rows per family (counts)")
display(summary)
failed = pd.concat([t[~t["passed"]].assign(family=name) for name, t in tables.items()])
failed if len(failed) else print("Every row passes.")

Check rows per family (counts)


,family,rows,passed,failed,rules (tolerance 0)
0,population,158,158,0,7
1,holdings,440,440,0,52
2,amount_rules,112,112,0,112
3,amount_levels,45,45,0,0
4,seasons,27,27,0,0
5,mart,82,82,0,82
6,traps,58,58,0,5
7,oracle,341,341,0,138


Every row passes.


## 2. Population
The mix should look like a UAE retail book: Mass dominant, Private small and old, ASIA dominant in
Mass, UAE nationals leading Private, digital onboarding more common lower down. Clients leave at
8–12% a year, a little more often lower down; entries exceed exits by the same margin everywhere,
so the mix stays flat.

In [3]:
desc = read_output(
    str(DATA),
    "mart",
    [
        "client_id",
        "snapshot_month",
        "subsegment",
        "segment",
        "nationality_group",
        "employment_status",
        "onboarding_channel",
        "income",
        "age",
        "salary_transfer_flag",
        "relationship_days",
        "residence_days",
    ],
    client_sample=5,
)
desc = desc.assign(profile=desc["subsegment"].map(PROFILE_OF))
first = desc[desc["snapshot_month"] == desc["snapshot_month"].min()]
print("Share of clients per subsegment, first panel month")
display(
    first["subsegment"]
    .value_counts(normalize=True)
    .reindex(ORDER)
    .round(4)
    .to_frame("share of clients")
)
print("Share of each level within the profile, first panel month")
for column in ["nationality_group", "employment_status", "onboarding_channel"]:
    display(
        pd.crosstab(first["profile"], first[column], normalize="index").reindex(PROFILES).round(2)
    )
print("Age in years, quantiles per subsegment, first panel month")
ages = first.groupby("subsegment")["age"].quantile([0.1, 0.5, 0.9]).unstack().reindex(ORDER)
display(ages.set_axis(["age p10", "age p50", "age p90"], axis=1))
print("Declared monthly income (AED) per subsegment, first panel month")
income = first.groupby("subsegment")["income"].agg(
    **{
        "median declared income": "median",
        "share without declared income": lambda s: s.isna().mean(),
    }
)
display(income.reindex(ORDER).round(2))

Share of clients per subsegment, first panel month


,share of clients
subsegment,
Upper Private,0.0022
Lower Private,0.0075
Upper Priority,0.0094
Lower Priority,0.0288
UA,0.0448
A,0.0417
UM,0.2008
MM,0.6217
LM,0.0431


Share of each level within the profile, first panel month


nationality_group,AFRICA,ARAB,ASIA,GCC,OTHER,UAE,WESTERN
profile,,,,,,,
private,0.03,0.12,0.16,0.12,0.04,0.39,0.14
priority,0.02,0.10,0.31,0.08,0.02,0.17,0.30
affluent,0.05,0.13,0.50,0.03,0.03,0.15,0.10
mass,0.07,0.12,0.74,0.01,0.01,0.04,0.02


employment_status,Employed,Other,Retired,Self-employed,Unemployed
profile,,,,,
private,0.23,0.14,0.19,0.34,0.10
priority,0.55,0.08,0.06,0.23,0.08
affluent,0.76,0.04,0.03,0.13,0.04
mass,0.91,0.01,0.01,0.05,0.01


onboarding_channel,branch,digital
profile,,
private,0.78,0.22
priority,0.70,0.30
affluent,0.54,0.46
mass,0.45,0.55


Age in years, quantiles per subsegment, first panel month


,age p10,age p50,age p90
subsegment,,,
Upper Private,39.0,52.5,66.0
Lower Private,38.0,53.0,71.7
Upper Priority,31.1,45.0,61.9
Lower Priority,30.0,42.0,57.0
UA,28.0,40.0,55.0
A,27.0,38.0,53.0
UM,26.0,36.0,49.0
MM,24.0,34.0,48.0
LM,23.0,32.0,45.0


Declared monthly income (AED) per subsegment, first panel month


,median declared income,share without declared income
subsegment,,
Upper Private,212060.0,0.67
Lower Private,87220.0,0.49
Upper Priority,58500.0,0.16
Lower Priority,42070.0,0.19
UA,32085.0,0.11
A,21230.0,0.09
UM,12160.0,0.05
MM,6040.0,0.06
LM,2950.0,0.04


In [4]:
# Turnover read from the mart, every client (a 20% sample is too noisy for the small profiles): a
# leaver's last row is its last month on the books, a new client's first row its first month after
# joining. Rates are shares of the profile's clients that month.
keys = read_output(str(DATA), "mart", ["client_id", "snapshot_month", "subsegment"])
keys = keys.assign(profile=keys["subsegment"].map(PROFILE_OF))
months = sorted(keys["snapshot_month"].unique())
book = keys.groupby(["snapshot_month", "profile"]).size()
last_rows = keys.drop_duplicates("client_id", keep="last")
first_rows = keys.drop_duplicates("client_id")
# A month without leavers (or joiners) counts as 0, not as missing.
exits = (
    last_rows[last_rows["snapshot_month"] < months[-1]]
    .groupby(["snapshot_month", "profile"])
    .size()
)
exits = exits.reindex(book.index, fill_value=0)
entries = (
    first_rows[first_rows["snapshot_month"] > months[0]]
    .groupby(["snapshot_month", "profile"])
    .size()
)
entries = entries.reindex(book.index, fill_value=0)
rates = pd.DataFrame({"exit rate": exits / book, "entry rate": entries / book}).reset_index()
rates = rates[(rates["snapshot_month"] > months[0]) & (rates["snapshot_month"] < months[-1])]
print("Mean monthly exit and entry rate per profile (share of the profile's clients), and the exit")
print("rate as a share leaving per year")
mean = rates.groupby("profile")[["exit rate", "entry rate"]].mean().reindex(PROFILES)
display(mean.assign(**{"leaving per year": 1 - (1 - mean["exit rate"]) ** 12}).round(4))
px.line(
    rates.melt(["snapshot_month", "profile"], ["exit rate", "entry rate"]),
    x="snapshot_month",
    y="value",
    color="profile",
    line_dash="variable",
    title="Monthly exit and entry rates per profile (share of the profile's clients)",
).show()

Mean monthly exit and entry rate per profile (share of the profile's clients), and the exit
rate as a share leaving per year


,exit rate,entry rate,leaving per year
profile,,,
private,0.0082,0.0084,0.0942
priority,0.0075,0.0107,0.0865
affluent,0.0090,0.0108,0.1030
mass,0.0111,0.0131,0.1251


## 3. Holdings and chains
Shares per profile should sit near the spec and stay flat (digital activity trends up by design:
the drift trap). Statuses split never / churned / active; loans mostly one account. By declared
income, the share holding a credit card, a personal or auto loan jumps at 5,000 AED and a home loan
at 15,000 AED: below a product's minimum income the odds of taking it are × 0.2 (clients without a
declared income count as above, if exempt).

In [5]:
flags = [
    "casa_flag",
    "casa_active_flag",
    "cc_flag",
    "cc_active_flag",
    "td_flag",
    "wealth_digital_flag",
    "wealth_trad_flag",
    "ebanking_flag",
    "digital_active_flag",
    "fx_active_flag",
]
hold = read_output(
    str(DATA),
    "mart",
    [
        "client_id",
        "snapshot_month",
        "subsegment",
        "income",
        *flags,
        "loan_personal_cnt",
        "loan_auto_cnt",
        "loan_home_cnt",
        "td_status",
        "product_cnt",
    ],
    client_sample=5,
)
hold = hold.assign(
    profile=hold["subsegment"].map(PROFILE_OF),
    **{f"{k}_held": hold[f"{k}_cnt"] > 0 for k in ["loan_personal", "loan_auto", "loan_home"]},
)
held = [*flags, "loan_personal_held", "loan_auto_held", "loan_home_held"]
ends = hold[hold["snapshot_month"].isin([months[0], months[-1]])]
print(
    "Share of clients holding each product (flags: share with the flag at 1; loans: share with at"
)
print("least one account), per profile, first and last panel month")
display(ends.groupby(["profile", "snapshot_month"])[held].mean().round(3))
monthly = hold.groupby(["snapshot_month", "profile"])[flags].mean().reset_index()
px.line(
    monthly.melt(["snapshot_month", "profile"], var_name="flag", value_name="share of clients"),
    x="snapshot_month",
    y="share of clients",
    color="flag",
    facet_col="profile",
    title="Share of clients with each flag at 1, per month and profile",
).show()
print("Term-deposit status: share of client-months")
display(hold["td_status"].value_counts(normalize=True).round(3).to_frame("share of client-months"))

Share of clients holding each product (flags: share with the flag at 1; loans: share with at
least one account), per profile, first and last panel month


casa_flag  casa_active_flag  cc_flag  cc_active_flag  td_flag  wealth_digital_flag  wealth_trad_flag  ebanking_flag  digital_active_flag  fx_active_flag  loan_personal_held  \
profile  snapshot_month                                                                                                                                                                                 
affluent 2024-10-01          0.992             0.932    0.652           0.509    0.150                0.114             0.085          0.882                0.708           0.294               0.205   
         2026-09-01          0.990             0.930    0.662           0.520    0.144                0.116             0.089          0.894                0.744           0.315               0.206   
mass     2024-10-01          0.989             0.951    0.299           0.217    0.041                0.020             0.010          0.746                0.584           0.118               0.202   
         2026-09-01          0.989             0.952    0.304           0.218    0.041                0.021             0.009          0.750                0.613           0.119               0.203   
priority 2024-10-01          0.989             0.925    0.783           0.642    0.312                0.185             0.329          0.899                0.714           0.500               0.117   
         2026-09-01          0.988             0.927    0.811           0.668    0.313                0.202             0.320          0.885                0.736           0.484               0.115   
private  2024-10-01          0.995             0.885    0.845           0.740    0.445                0.095             0.575          0.825                0.630           0.590               0.045   
         2026-09-01          0.995             0.915    0.839           0.688    0.432                0.126             0.573          0.794                0.658           0.678               0.045   

                         loan_auto_held  loan_home_held  
profile  snapshot_month                                  
affluent 2024-10-01               0.122           0.111  
         2026-09-01               0.127           0.107  
mass     2024-10-01               0.058           0.020  
         2026-09-01               0.062           0.018  
priority 2024-10-01               0.089           0.223  
         2026-09-01               0.093           0.217  
private  2024-10-01               0.035           0.175  
         2026-09-01               0.035           0.191

Term-deposit status: share of client-months


,share of client-months
td_status,
never,0.843
churned,0.093
active,0.064


In [6]:
# Share holding each product with an income threshold, by declared income (first panel month).
thresholds = {key: p["min_income"] for key, p in spec["products"].items() if "min_income" in p}
declared = hold[(hold["snapshot_month"] == months[0]) & hold["income"].notna()]
edges = np.geomspace(1_000, 200_000, 31)
band = pd.cut(declared["income"], edges)
columns = {
    "cc": "cc_flag",
    "loan_personal": "loan_personal_held",
    "loan_auto": "loan_auto_held",
    "loan_home": "loan_home_held",
}
by_income = declared.groupby(band, observed=True)[list(columns.values())].mean()
by_income.index = [interval.mid for interval in by_income.index]
fig = px.line(
    by_income.rename(columns={v: k for k, v in columns.items()}),
    log_x=True,
    markers=True,
    labels={"index": "declared monthly income (AED, log scale)", "value": "share holding"},
    title="Share holding each product by declared income; dotted lines: minimum incomes",
)
for minimum in sorted(set(thresholds.values())):
    fig.add_vline(x=minimum, line_dash="dot", annotation_text=f"{minimum:,} AED")
fig.show()
print("Minimum declared income per product (AED):", thresholds)

Minimum declared income per product (AED): {'cc': 5000, 'loan_personal': 5000, 'loan_auto': 5000, 'loan_home': 15000}


## 4. Amounts
Medians ordered by subsegment, heavy right tails, Private far above everyone else. Card limits are
a few product tiers: the largest tier at or below the profile's multiple of income (3× Private to
1.5× Mass), one tier up or down for a few new cards. Revolvers sit at higher utilisation than
transactors but rarely exactly at the limit.

In [7]:
amount_cols = [
    "casa_bal",
    "inflow_casa_vol",
    "outflow_casa_vol",
    "spend_cc_vol",
    "cc_limit",
    "cc_utilisation_ratio",
    "td_bal",
    "wealth_trad_bal",
    "wealth_digital_bal",
    "loan_personal_bal",
    "loan_home_bal",
    "fx_vol",
    "login_digital_mobile_cnt",
]
amt = read_output(
    str(DATA),
    "mart",
    ["client_id", "snapshot_month", "subsegment", "income", *amount_cols],
    client_sample=5,
)
may = amt[amt["snapshot_month"] == MAY]
print("Median of each amount among clients with a value above 0, May 2025, per subsegment")
print("(AED; utilisation a ratio; logins a count)")
medians = may.groupby("subsegment")[amount_cols].agg(lambda s: s[s > 0].median()).reindex(ORDER)
display(medians.round(2).rename(columns=lambda c: f"median {c}"))
counts, edges = np.histogram(np.log10(may.loc[may["casa_bal"] > 0, "casa_bal"]), bins=60)
px.bar(
    x=edges[:-1],
    y=counts,
    labels={"x": "log10 current/savings balance (AED)", "y": "clients"},
    title="Current/savings balance above 0, May 2025: clients per bin (20% sample)",
).show()

Median of each amount among clients with a value above 0, May 2025, per subsegment
(AED; utilisation a ratio; logins a count)


,median casa_bal,median inflow_casa_vol,median outflow_casa_vol,median spend_cc_vol,median cc_limit,median cc_utilisation_ratio,median td_bal,median wealth_trad_bal,median wealth_digital_bal,median loan_personal_bal,median loan_home_bal,median fx_vol,median login_digital_mobile_cnt
subsegment,,,,,,,,,,,,,
Upper Private,4339692.79,268474.53,346385.76,67819.67,250000.0,0.22,4187721.77,12882591.85,2731345.79,199077.06,9096683.64,210471.74,6.0
Lower Private,2085066.10,166409.77,111446.48,26158.54,150000.0,0.20,1308706.09,7872837.78,730047.45,265667.25,4152997.46,82678.92,5.0
Upper Priority,153217.79,69538.18,64499.79,13675.45,100000.0,0.19,538751.79,1104950.54,301896.40,449858.70,4061771.70,37610.14,10.0
Lower Priority,100096.22,45613.92,45620.72,9949.06,75000.0,0.19,442243.51,649312.51,159559.30,185009.42,2833127.65,22359.64,9.0
UA,34218.55,37372.76,36903.17,8237.54,50000.0,0.24,210944.87,252450.00,72872.78,151878.70,2013466.24,12487.93,14.0
A,23702.64,23793.00,23925.90,4750.22,35000.0,0.27,124303.87,172722.50,66100.53,107069.92,1322452.68,9093.28,14.0
UM,4987.12,13507.67,13333.22,2851.97,10000.0,0.48,35033.94,54094.66,15124.94,67258.63,883555.40,3731.14,20.0
MM,2447.11,6343.88,6342.78,1579.23,10000.0,0.49,20712.45,31324.40,7135.93,40229.72,354875.87,2011.24,20.0
LM,1225.92,2938.90,2997.39,774.12,5000.0,0.38,10000.00,14363.19,2016.43,18069.54,145873.00,1205.93,21.0


In [8]:
# Card limits against income: each card's limit is a product tier, set from the profile's multiple
# of true income (declared income shown; clients without one are left out).
holders = may[(may["cc_limit"] > 0) & may["income"].notna()].assign(
    profile=lambda d: d["subsegment"].map(PROFILE_OF)
)
px.scatter(
    holders.sample(min(len(holders), 4_000), random_state=0),
    x="income",
    y="cc_limit",
    color="profile",
    log_x=True,
    log_y=True,
    opacity=0.5,
    labels={"income": "declared monthly income (AED)", "cc_limit": "card limit (AED)"},
    title="Card limit vs declared income, May 2025 (one dot a card; the bands are the tiers)",
).show()
print("Card limit tiers (AED):", spec["levels"]["card"]["limit_tiers"])
print("Share of cards per limit tier, per profile, May 2025")
display(
    pd.crosstab(holders["profile"], holders["cc_limit"], normalize="index")
    .reindex(PROFILES)
    .round(3)
)
hidden = read_output(
    str(DATA), "oracle", ["client_id", "snapshot_month", "revolver_flag"], client_sample=5
)
cards = may.merge(hidden, on=["client_id", "snapshot_month"])
cards = cards[cards["cc_limit"] > 0].assign(
    kind=lambda d: np.where(d["revolver_flag"] == 1, "revolver", "transactor")
)
bins = np.linspace(0, 1, 21)
util = pd.DataFrame(
    {
        kind: np.histogram(part["cc_utilisation_ratio"], bins=bins)[0] / len(part)
        for kind, part in cards.groupby("kind")
    },
    index=bins[:-1].round(2),
)
px.bar(
    util,
    barmode="group",
    labels={"index": "utilisation (balance / limit)", "value": "share of cards"},
    title="Card utilisation, revolvers vs transactors, May 2025 (share of each kind's cards)",
).show()

Card limit tiers (AED): [5000, 10000, 20000, 35000, 50000, 75000, 100000, 150000, 250000, 500000]
Share of cards per limit tier, per profile, May 2025


cc_limit,5000.0,10000.0,20000.0,35000.0,50000.0,75000.0,100000.0,150000.0,250000.0,500000.0
profile,,,,,,,,,,
private,0.000,0.000,0.000,0.012,0.047,0.023,0.058,0.256,0.349,0.256
priority,0.002,0.008,0.017,0.040,0.138,0.197,0.299,0.213,0.084,0.002
affluent,0.003,0.037,0.165,0.246,0.295,0.156,0.079,0.020,0.001,0.000
mass,0.346,0.462,0.159,0.026,0.005,0.001,0.000,0.000,0.000,0.000


In [9]:
flow_cols = [
    "inflow_casa_vol",
    "outflow_casa_vol",
    "casa_bal",
    "casa_active_flag",
    "outflow_casa_remittance_vol",
    "outflow_casa_cash_vol",
    "outflow_casa_bills_vol",
    "outflow_casa_transfers_vol",
    "spend_cc_travel_vol",
    "spend_cc_retail_vol",
    "spend_cc_online_vol",
    "spend_cc_dining_vol",
    "nationality_group",
]
flows = read_output(
    str(DATA), "mart", ["client_id", "snapshot_month", "subsegment", *flow_cols], client_sample=5
).sort_values(["client_id", "snapshot_month"])
# Rows with a previous row in the panel (a starting-book client's first panel row continues the
# burn-in).
change = flows["casa_bal"] - flows.groupby("client_id")["casa_bal"].shift()
gap = (flows["inflow_casa_vol"] - flows["outflow_casa_vol"] - change)[
    flows["casa_active_flag"] == 1
]
print(f"booking identity, largest |inflow − outflow − balance change| (AED): {gap.abs().max():.4f}")
out_parts = [
    "outflow_casa_remittance_vol",
    "outflow_casa_cash_vol",
    "outflow_casa_bills_vol",
    "outflow_casa_transfers_vol",
]
card_parts = [
    "spend_cc_travel_vol",
    "spend_cc_retail_vol",
    "spend_cc_online_vol",
    "spend_cc_dining_vol",
]
prof = flows.assign(profile=flows["subsegment"].map(PROFILE_OF))
print("Share of each subcategory in the profile's total outflow (all panel months)")
total = prof.groupby("profile")[out_parts].sum()
display(total.div(total.sum(axis=1), axis=0).reindex(PROFILES).round(3))
print("Share of each subcategory in the profile's total card spend (all panel months)")
total = prof.groupby("profile")[card_parts].sum()
display(total.div(total.sum(axis=1), axis=0).reindex(PROFILES).round(3))
print("Remittance per nationality group: share of client-months with a remittance, and the mean")
print("remittance per client-month (AED, zeros included)")
display(
    flows.groupby("nationality_group")["outflow_casa_remittance_vol"]
    .agg(**{"share remitting": lambda s: (s > 0).mean(), "mean remittance": "mean"})
    .round(2)
)

booking identity, largest |inflow − outflow − balance change| (AED): 0.0000
Share of each subcategory in the profile's total outflow (all panel months)


,outflow_casa_remittance_vol,outflow_casa_cash_vol,outflow_casa_bills_vol,outflow_casa_transfers_vol
profile,,,,
private,0.002,0.008,0.039,0.951
priority,0.016,0.050,0.164,0.770
affluent,0.051,0.115,0.225,0.608
mass,0.133,0.253,0.226,0.388


Share of each subcategory in the profile's total card spend (all panel months)


,spend_cc_travel_vol,spend_cc_retail_vol,spend_cc_online_vol,spend_cc_dining_vol
profile,,,,
private,0.314,0.307,0.194,0.185
priority,0.179,0.368,0.246,0.207
affluent,0.084,0.439,0.281,0.197
mass,0.009,0.514,0.270,0.206


Remittance per nationality group: share of client-months with a remittance, and the mean
remittance per client-month (AED, zeros included)


,share remitting,mean remittance
nationality_group,,
AFRICA,0.55,1456.21
ARAB,0.55,1397.13
ASIA,0.56,1524.92
GCC,0.00,0.00
OTHER,0.51,1397.41
UAE,0.00,0.00
WESTERN,0.46,1013.30


## 5. Time effects
Summer: travel spend and FX up, balances down, less so for Mass. School fees in September,
January and April. Bonuses January–April. Mobile logins jump at the app relaunch (2026-01).

In [10]:
seasonal = (
    prof.groupby(["snapshot_month", "profile"])
    .agg(
        **{
            "mean travel spend": ("spend_cc_travel_vol", "mean"),
            "mean bills": ("outflow_casa_bills_vol", "mean"),
            "median balance": ("casa_bal", "median"),
        }
    )
    .reset_index()
)
for column in ["mean travel spend", "mean bills", "median balance"]:
    px.line(
        seasonal,
        x="snapshot_month",
        y=column,
        color="profile",
        log_y=True,
        title=f"{column} per client-month (AED), by month and profile",
    ).show()
logins = amt.groupby("snapshot_month")["login_digital_mobile_cnt"].apply(lambda s: s[s > 0].mean())
px.line(
    logins.reset_index(),
    x="snapshot_month",
    y="login_digital_mobile_cnt",
    labels={"login_digital_mobile_cnt": "mean mobile logins"},
    title="Mean mobile logins a month of clients with any (drift at 2026-01)",
).show()

## 6. Segments overlap; Simpson's reversal
About a quarter of MM clients hold more than the UM median; Private far apart. Logins vs card
acquisition: positive within each onboarding channel, negative pooled.

In [11]:
mm_um = may[may["subsegment"].isin(["UM", "MM"]) & (may["casa_bal"] > 0)]
bins = np.linspace(1, 7, 61)
hist = pd.DataFrame(
    {
        s: np.histogram(np.log10(part["casa_bal"]), bins=bins)[0] / len(part)
        for s, part in mm_um.groupby("subsegment")
    },
    index=bins[:-1].round(1),
)
px.line(
    hist,
    labels={"index": "log10 current/savings balance (AED)", "value": "share of clients"},
    title="Current/savings balance, UM vs MM, May 2025 (share of each subsegment's clients)",
).show()
um_median = mm_um.loc[mm_um["subsegment"] == "UM", "casa_bal"].median()
above = (mm_um.loc[mm_um["subsegment"] == "MM", "casa_bal"] > um_median).mean()
print(f"share of MM clients above the UM median balance: {above:.3f}")

share of MM clients above the UM median balance: 0.244


In [12]:
sim_cols = [
    "client_id",
    "snapshot_month",
    "cc_flag",
    "onboarding_channel",
    "subsegment",
    "relationship_days",
    "login_digital_mobile_cnt_0_2_months_avg",
]
sim_mart = read_output(str(DATA), "mart", None, client_sample=5)
sim_oracle = read_output(str(DATA), "oracle", None, client_sample=5)
chance = window_probability(sim_mart, sim_oracle, truth, "cc_acquire", 3)
frame = sim_mart[sim_cols].assign(chance=chance)
frame = frame[(frame["cc_flag"] == 0) & frame["chance"].notna()]
frame = frame.assign(
    logins=pd.qcut(
        frame["login_digital_mobile_cnt_0_2_months_avg"].rank(method="first"), 10, labels=False
    )
    + 1
)
pooled = frame.groupby("logins")["chance"].mean().rename("pooled")
within = frame.groupby(["onboarding_channel", "logins"])["chance"].mean().unstack(0)
px.line(
    pd.concat([pooled, within], axis=1).reset_index(),
    x="logins",
    y=["pooled", "branch", "digital"],
    labels={"logins": "decile of mobile logins (3-month average)", "value": "mean chance"},
    title="Mean chance of taking a card within 3 months (non-holders), by login decile",
).show()

## 7. Mart columns
147 columns from one grammar; nulls only where the retail data has them; rollups null until the
window fits inside the relationship. A null shows as `<NA>` in integer columns and `NaN` in float
columns: both are true nulls (pandas' nullable integers display missing values as `<NA>`); the
parquet files store them as nulls, and Spark reads them as `null` (`tests/test_synthetic_mart.py`).

In [13]:
schema = pd.DataFrame(truth["columns"])
print("Mart columns per kind (counts)")
display(schema.groupby("kind").size().to_frame("columns"))
nulls = sim_mart.isna().mean()
print("Share of rows that are null, per column with any null (20% sample, all months)")
display(nulls[nulls > 0].sort_values(ascending=False).round(3).to_frame("share null"))

Mart columns per kind (counts)


,columns
kind,
categorical,7
count,8
flow,20
holding,20
id,1
rollup,65
static,5
stock,10
time,1


Share of rows that are null, per column with any null (20% sample, all months)


,share null
legacy_risk_grade_trap_all_null,1.000
loan_home_maturity_months,0.964
loan_auto_maturity_months,0.933
loan_personal_maturity_months,0.802
months_since_last_fx,0.748
spend_cc_vol_0_2_vs_3_5_months_ratio,0.719
cc_utilisation_ratio,0.643
cc_utilisation_ratio_0_5_months_avg,0.639
cc_utilisation_ratio_0_2_months_avg,0.639
months_since_last_spend_cc,0.601


In [14]:
# One client with a card who spends in some months only, so its recency column takes values.
per_client = sim_mart.groupby("client_id").agg(
    months=("snapshot_month", "size"), spending=("spend_cc_vol", lambda s: (s > 0).mean())
)
example_id = per_client[
    (per_client["months"] == 24) & per_client["spending"].between(0.4, 0.8)
].index[0]
example = sim_mart[sim_mart["client_id"] == example_id]
print(f"Client {example_id}: monthly average balance (AED), its 3- and 6-month rollups, card spend")
print("(AED) and months since the last month with card spend (0 = spent this month)")
display(
    example[
        [
            "snapshot_month",
            "casa_bal",
            "casa_bal_0_2_months_avg",
            "casa_bal_0_5_months_avg",
            "casa_bal_0_5_months_std",
            "spend_cc_vol",
            "months_since_last_spend_cc",
            "relationship_days",
        ]
    ]
)
column = "months_since_last_spend_cc"
values = sim_mart[column]
print(
    f"{column}: pandas dtype {values.dtype}; {values.isna().sum():,} nulls; "
    f"{values.map(lambda v: isinstance(v, str)).sum()} string values"
)
part = pq.read_table(
    next((DATA / "mart").glob("snapshot_month=*/part-0.parquet")), columns=[column]
)
stored, nulls = part.schema.field(column).type, part.column(column).null_count
print(f"in the parquet file: type {stored}, {nulls:,} nulls of {part.num_rows:,} rows (one month)")

Client 5: monthly average balance (AED), its 3- and 6-month rollups, card spend
(AED) and months since the last month with card spend (0 = spent this month)


,snapshot_month,casa_bal,casa_bal_0_2_months_avg,casa_bal_0_5_months_avg,casa_bal_0_5_months_std,spend_cc_vol,months_since_last_spend_cc,relationship_days
0,2024-10-01,5853.33,5333.283333,5507.568333,499.695106,1609.61,0,754
20528,2024-11-01,5799.24,5703.560000,5451.615000,429.218686,2070.27,0,784
41114,2024-12-01,7310.61,6321.060000,5732.851667,880.114022,2665.44,0,815
61739,2025-01-01,5839.87,6316.573333,5824.928333,852.662820,1071.77,0,846
82394,2025-02-01,6071.49,6407.323333,6055.441667,645.828116,4383.82,0,874
103111,2025-03-01,6575.87,6162.410000,6241.735000,598.544747,0.00,1,905
123876,2025-04-01,5350.17,5999.176667,6157.875000,691.825669,0.00,2,935
144692,2025-05-01,4590.05,5505.363333,5956.343333,946.454423,0.00,3,966
165560,2025-06-01,5364.53,5101.583333,5631.996667,687.587104,0.00,4,996
186469,2025-07-01,5516.41,5156.996667,5578.086667,680.674864,0.00,5,1027


months_since_last_spend_cc: pandas dtype Int64; 304,192 nulls; 0 string values
in the parquet file: type int64, 63,692 nulls of 104,252 rows (one month)


## 8. Traps
The trap summary, then the evidence. `trap_column` True: a column added only to test the
framework, its name ending in `_trap_<kind>`. False: the trap sits in real columns (named in
`column`; none for collinearity and dirty data), which keep their real names because the framework
meets the same columns in the real mart. The leaks beat the oracle; the shifted column holds next
month's personal-loan balance; the null controls carry nothing.

In [15]:
print("The trap summary (truth.json: spec.traps)")
display(trap_summary(spec)[["product", "kind", "trap_column", "column", "expected"]])
trap_rows = tables["traps"]
print("Evidence: the trap family's rows for the leaks, the upstream target, the shifted column,")
print("the rare level, Simpson's reversal, the null controls and the drift")
display(
    trap_rows[
        trap_rows["check"].str.contains(
            "leak|upstream|shifted|Ghana|Simpson|null control|PSI", case=False
        )
    ][["check", "stratum", "expected", "realised", "tolerance", "passed"]]
)

The trap summary (truth.json: spec.traps)


,product,kind,trap_column,column,expected
0,cc,leak,True,cc_preapproval_score_trap_leak,univariate AUC ≈ 0.97 for card acquisition in ...
1,cc,leak,True,cc_campaign_response_score_trap_leak,univariate AUC ≈ 0.875 for card acquisition in...
2,cc,upstream_target,True,cc_acquire_next_3_months_flag_trap_upstream_ta...,equals Target Factory's own target except leav...
3,cc,simpson,False,login_digital_mobile_cnt,logins vs card acquisition: positive within ea...
4,cc,null_income_eligibility,False,income,a plain income filter silently drops exempt cl...
5,cc,onboarding,False,relationship_days,most adoption in the first 3 months: the stron...
6,cc,eligibility_defining,False,spend_cc_vol,"constant (0) within non-holders, the eligible ..."
7,cc,decimal_type,False,cc_limit,Spark reads a decimal; pandas gets Python obje...
8,loan_personal,process_leak,True,application_loan_personal_cnt_trap_process_leak,beats the oracle for openings in the next month
9,loan_personal,shifted,True,loan_personal_bal_trap_shifted,holds next month's loan_personal_bal (an as-of...


Evidence: the trap family's rows for the leaks, the upstream target, the shifted column,
the rare level, Simpson's reversal, the null controls and the drift


,check,stratum,expected,realised,tolerance,passed
0,leak cc_preapproval_score_trap_leak: AUC for c...,"w=3, one month",0.970000,0.966914,0.012336,True
1,leak cc_preapproval_score_trap_leak: AUC above...,"w=3, one month",0.841854,0.966914,NaN,True
2,leak cc_campaign_response_score_trap_leak: AUC...,"w=3, one month",0.875000,0.876865,0.023346,True
3,leak cc_campaign_response_score_trap_leak: AUC...,"w=3, one month",0.841854,0.876865,NaN,True
4,leaks: correlation among non-acquirers (indepe...,"w=3, one month",0.000000,0.007170,0.017838,True
5,upstream: equals the realised cc_acquire (w=3),all,0.000000,0.000000,0.000000,True
6,upstream: 0 for leavers without a card,all,0.000000,0.000000,0.000000,True
7,upstream: null exactly for holders and the las...,all,0.000000,0.000000,0.000000,True
8,process leak: openings at t+1 with an applicat...,all,0.800000,0.800000,0.013910,True
9,process leak: applications at t declined (no o...,all,0.300000,0.303090,0.014874,True


## 9. Oracle
The ceiling per target (oracle AUC, windows of 1, 3 and 6 months), calibration per decile for card
acquisition, regression coverage; then single-column AUCs for the activation targets: how far each
mart column alone gets, against the oracle.

In [16]:
oracle_auc = pd.DataFrame(
    {
        t: {
            "oracle AUC w=1": m["w1"]["auc"],
            "oracle AUC w=3": m["w3"]["auc"],
            "oracle AUC w=6": m["w6"]["auc"],
            "positive rows a month (w=3)": m["w3"]["positive_rows_per_month"],
            "expected events in 6 months": m["expected_events_6m"],
            "low confidence": m["low_confidence"],
        }
        for t, m in truth["oracle"].items()
    }
).T
print("Per standard target: the oracle's AUC (all rows, all months), positive rows a month, sizing")
display(oracle_auc.round(3))
y = realised_target(sim_mart, "cc_acquire", 3)
cal = pd.DataFrame({"window probability": chance, "realised": y}).dropna()
cal = cal.groupby(pd.qcut(cal["window probability"].rank(method="first"), 10, labels=False)).mean()
px.scatter(
    cal,
    x="window probability",
    y="realised",
    title="cc_acquire (w=3): realised rate vs mean window probability, per probability decile",
).show()
oracle_rows = tables["oracle"]
print(
    "Regression oracle (next-3-month card spend): coverage of its quantiles and the ratio of means"
)
display(
    oracle_rows[oracle_rows["check"].str.contains("spend_cc_next_3_months", case=False)][
        ["check", "stratum", "expected", "realised", "tolerance", "passed"]
    ].head(12)
)

Per standard target: the oracle's AUC (all rows, all months), positive rows a month, sizing


,oracle AUC w=1,oracle AUC w=3,oracle AUC w=6,positive rows a month (w=3),expected events in 6 months,low confidence
cc_acquire,0.866691,0.845755,0.833094,1901.333333,4716.960857,False
td_acquire,0.747465,0.744742,0.742579,826.333333,1664.684716,False
wealth_digital_acquire,0.794901,0.793363,0.790511,1245.904762,2579.641739,False
wealth_trad_acquire,0.885013,0.882011,0.880958,134.142857,276.745602,False
ebanking_acquire,0.950727,0.925684,0.899609,904.857143,3211.555618,False
loan_personal_open,0.765953,0.765965,0.767836,2105.47619,4360.178615,False
loan_auto_open,0.717806,0.71551,0.714081,505.857143,1023.625829,False
loan_home_open,0.829866,0.828261,0.827623,139.714286,291.628521,False
fx_activate,0.744419,0.742687,0.739642,1736.571429,3540.436233,False
digital_activate,0.826436,0.794977,0.799004,5140.333333,15930.131098,False


Regression oracle (next-3-month card spend): coverage of its quantiles and the ratio of means


,check,stratum,expected,realised,tolerance,passed
294,spend_cc_next_3_months: P(realised < q10),observation 2024-11,0.111314,0.078237,NaN,True
295,"spend_cc_next_3_months: P(realised < q10), q10...",observation 2024-11,0.103980,0.104366,0.008471,True
296,"spend_cc_next_3_months: P(realised ≤ q50), q50...",observation 2024-11,0.500000,0.501349,0.013493,True
297,spend_cc_next_3_months: P(realised ≤ q90),observation 2024-11,0.888686,0.896273,NaN,True
298,spend_cc_next_3_months: realised / predicted mean,observation 2024-11,1.000000,0.998102,0.024531,True
299,spend_cc_next_3_months: P(realised < q10),observation 2024-12,0.111307,0.075973,NaN,True
300,"spend_cc_next_3_months: P(realised < q10), q10...",observation 2024-12,0.103980,0.101271,0.008459,True
301,"spend_cc_next_3_months: P(realised ≤ q50), q50...",observation 2024-12,0.500000,0.502010,0.013479,True
302,spend_cc_next_3_months: P(realised ≤ q90),observation 2024-12,0.888693,0.898902,NaN,True
303,spend_cc_next_3_months: realised / predicted mean,observation 2024-12,1.000000,0.994191,0.023659,True


In [17]:
# Single-column AUCs: every numeric mart column alone against the realised target (w = 3), in the
# middle panel month (one row per client), next to the oracle's window probability on the same rows.
# "strength" is max(AUC, 1 − AUC): a column can predict either way round. Columns with a value in
# fewer than 300 eligible rows are left out: their AUC is mostly noise.
observation = sorted(sim_mart["snapshot_month"].unique())[11]
at_obs = (sim_mart["snapshot_month"] == observation).to_numpy()
numeric = [
    c
    for c in sim_mart.columns
    if c not in ("client_id", "snapshot_month") and pd.api.types.is_numeric_dtype(sim_mart[c])
]
rows = []
for target in ["cc_activate", "digital_activate", "casa_reactivate", "fx_activate"]:
    y = realised_target(sim_mart, target, 3)
    use = at_obs & y.notna().to_numpy()
    labels = y[use].to_numpy()
    probability = window_probability(sim_mart, sim_oracle, truth, target, 3).to_numpy()[use]
    rows.append(
        {
            "target": target,
            "column": "ORACLE (window probability)",
            "AUC": auc(probability, labels),
            "rows": int(use.sum()),
            "events": int(labels.sum()),
        }
    )
    for column in numeric:
        values = sim_mart.loc[use, column].to_numpy(dtype=float, na_value=np.nan)
        known = ~np.isnan(values)
        if known.sum() >= 300 and np.unique(values[known]).size > 1:  # sparse columns: noise
            rows.append(
                {
                    "target": target,
                    "column": column,
                    "AUC": auc(values[known], labels[known]),
                    "rows": int(known.sum()),
                    "events": int(labels[known].sum()),
                }
            )
single = pd.DataFrame(rows).assign(strength=lambda d: np.maximum(d["AUC"], 1 - d["AUC"]))
print(f"Top 8 single columns per activation target by strength, {observation:%Y-%m}, 20% sample")
display(
    single.sort_values(["target", "strength"], ascending=[True, False])
    .groupby("target")
    .head(9)
    .round(3)
    .set_index(["target", "column"])
)

Top 8 single columns per activation target by strength, 2025-09, 20% sample


AUC   rows  events  strength
target           column                                                                 
casa_reactivate  ORACLE (window probability)              0.679    864     305     0.679
                 inflow_casa_salary_vol_0_5_months_avg    0.601    809     280     0.601
                 salary_transfer_flag                     0.600    864     305     0.600
                 outflow_casa_cash_vol_0_5_months_avg     0.594    809     280     0.594
                 inflow_casa_vol_0_5_months_avg           0.591    809     280     0.591
                 outflow_casa_vol_0_5_months_avg          0.590    809     280     0.590
                 inflow_casa_vol_0_5_months_max           0.590    809     280     0.590
                 inflow_casa_vol_0_5_months_std           0.589    809     280     0.589
                 outflow_casa_bills_vol_0_5_months_avg    0.587    809     280     0.587
cc_activate      ORACLE (window probability)              0.702   1761     682     0.702
                 inflow_casa_transfers_vol                0.580   1761     682     0.580
                 inflow_casa_vol_0_5_months_std           0.572   1616     605     0.572
                 inflow_casa_other_vol_0_2_months_avg     0.571   1685     635     0.571
                 inflow_casa_other_vol                    0.571   1761     682     0.571
                 outflow_casa_cnt_0_5_months_avg          0.571   1616     605     0.571
                 inflow_casa_other_vol_0_5_months_avg     0.569   1616     605     0.569
                 inflow_casa_vol_0_5_months_max           0.557   1616     605     0.557
                 outflow_casa_bills_vol_0_5_months_avg    0.555   1616     605     0.555
digital_activate ORACLE (window probability)              0.801   3220    1064     0.801
                 days_since_last_login_digital            0.389   2653     796     0.611
                 login_digital_mobile_cnt_0_5_months_std  0.609   2905     857     0.609
                 login_digital_mobile_cnt_0_5_months_max  0.608   2905     857     0.608
                 login_digital_mobile_cnt_0_5_months_avg  0.606   2905     857     0.606
                 login_digital_online_cnt_0_5_months_avg  0.595   2905     857     0.595
                 relationship_days                        0.409   3220    1064     0.591
                 login_digital_mobile_cnt_0_5_months_nz   0.589   2905     857     0.589
                 residence_days                           0.425   3046    1015     0.575
fx_activate      ORACLE (window probability)              0.737  17268     330     0.737
                 income                                   0.626  16197     297     0.626
                 casa_bal_0_5_months_min                  0.617  16194     311     0.617
                 casa_bal                                 0.616  17268     330     0.616
                 casa_bal_0_5_months_avg                  0.616  16194     311     0.616
                 casa_bal_0_5_months_max                  0.616  16194     311     0.616
                 casa_bal_0_2_months_avg                  0.615  16823     322     0.615
                 inflow_casa_vol_0_5_months_avg           0.601  16194     311     0.601
                 outflow_casa_vol_0_5_months_avg          0.601  16194     311     0.601

## 10. Client trajectories and hidden traits
A few clients month by month: balances, spend, flags, and the hidden traits behind them.

In [18]:
some = sim_mart["client_id"].drop_duplicates().sample(3, random_state=0)
path = sim_mart[sim_mart["client_id"].isin(some)]
px.line(
    path,
    x="snapshot_month",
    y="casa_bal",
    color="client_id",
    log_y=True,
    title="Monthly average current/savings balance (AED)",
).show()
px.line(
    path, x="snapshot_month", y="spend_cc_vol", color="client_id", title="Card spend a month (AED)"
).show()
traits = sim_oracle[sim_oracle["client_id"].isin(some)].melt(
    ["client_id", "snapshot_month"], [c for c in sim_oracle.columns if c.startswith("z_")]
)
px.line(
    traits,
    x="snapshot_month",
    y="value",
    color="variable",
    facet_row="client_id",
    title="Hidden traits (oracle; standardised scale)",
).show()